In [ ]:
import logging

FORMAT = '%(asctime)s %(name)s %(funcName)s %(message)s'
log_level = logging.WARNING
logging.basicConfig(format=FORMAT, datefmt='%H:%M:%S',
                    level=log_level)

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import h5py
import numpy as np
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import csv
from pathlib import Path 

bnsi_path = '/scicore/home/nimwegen/degroo0000/Bonsai-data-representation'
sys.path.append(bnsi_path)
from bonsai_scout.bonsai_scout_helpers import Bonvis_figure, Bonvis_settings, Bonvis_metadata

#### Set some parameters

In [ ]:
SINGLE_DATASET = True
N_PCS_UMAP = -1
N_PCS_PHATE = 100
N_PCS_DTNE = 500
methods = ['truth', 'bonsai', 'pca', 'umap', 'phate', 'DTNE']
# methods = ['bonsai', 'pca', 'umap', 'sanity']

SAVE_FIG = True


#### List information on datasets

In [ ]:
if not SINGLE_DATASET:
    dataset_ids = [
        'simulated_datasets/random_numbers_512genes_1024cells_seed2462',
        'simulated_datasets/random_numbers_512genes_1024cells_seed2462_nosanity',
    ]
else:
    dataset_ids = ['simulated_datasets/random_numbers_512genes_1024cells_seed2462_nosanity']
    # dataset_ids = ['simulated_datasets/simulated_pseudobulk_based_ncells_1024_seed_1231']
    
input_data_folders = [os.path.join('/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets', dataset_id, 'UMI_counts') for dataset_id in dataset_ids]
bonsai_results_folders = [os.path.join('/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets', dataset_id, 'bonsai') 
                          for dataset_id in dataset_ids]

if not SINGLE_DATASET:
    dataset_descr_lst = ['Random integers', 'Random integers (no scRNA-seq assumption)']
else:
    dataset_descr_lst = ['Random integers']
    # dataset_descr_lst = ['Pseudobulk']

## Create Bonsai visualization of dataset

In [ ]:
%%capture  
# The above %%capture is used for not showing the tree-visualizations that are created.
bonvis_metadata_lst = []
bonvis_settings_lst = []
bonvis_data_hdf_lst = []
bonvis_fig_lst = []
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    # Load metadata, settings and data
    data_path = os.path.join(bonsai_results, 'bonsai_vis_data.hdf')
    settings_path = os.path.join(bonsai_results, 'bonsai_vis_settings.json')
    bonvis_metadata = Bonvis_metadata(data_path)
    bonvis_settings = Bonvis_settings(load_settings_path=settings_path)
    bonvis_data_hdf = h5py.File(data_path, 'r')
    bonvis_fig = Bonvis_figure(bonvis_data_hdf, bonvis_metadata, bonvis_data_path=data_path,
                           bonvis_settings=bonvis_settings)
    bonvis_fig.create_figure(figsize=(6, 6))

    bonvis_metadata_lst.append(bonvis_metadata)
    bonvis_settings_lst.append(bonvis_settings)
    bonvis_data_hdf_lst.append(bonvis_data_hdf)
    bonvis_fig_lst.append(bonvis_fig)

In [ ]:
%%capture  
# It is possible to create circular layout
for ind_dataset, bonvis_fig in enumerate(bonvis_fig_lst):
    bonvis_fig.update_figure(ly_type='ly_eq_angle', scale_edges=1.25)
    bonvis_fig.bonvis_settings.node_style['annot_info'].annot_to_color['default'] = (0.20, 0.55, 0.50, 1.0)
    bonvis_fig.create_figure(figsize=(6, 6))

In [ ]:
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_fig_lst):
    geometry = 'flat'
    zoom = 1
    bonvis_fig.update_figure(ly_type='ly_eq_angle', geometry=geometry, zoom=zoom);

## Create PCA and UMAP plot of all datasets

In [ ]:
from paper_figure_scripts_and_notebooks.simulating_datasets.analyzing_simulated_datasets.knn_recall_helpers import do_pca, do_logp1, fit_umap, fit_phate, \
    fit_DTNE, get_pdists_on_tree, Dataset, \
    compare_nearest_neighbours_to_truth, compare_pdists_to_truth, get_pdists_on_tree, compare_pdists_to_truth_per_cell, compare_nearest_neighbours_to_truth
from bonsai.bonsai_helpers import find_latest_tree_folder
from scipy.spatial import distance
from bonsai.bonsai_dataprocessing import SCData, get_bonsai_euclidean_distances
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from matplotlib import colormaps
plt.set_loglevel(level='warning')

In [ ]:
umi_counts_lst = []
cell_ids_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders):    
    print(ind_dataset)
    # Load data; extract UMI-counts and cell-IDs
    umi_counts_df = pd.read_csv(os.path.join(input_data_folder, 'Gene_table.txt'), header=0,
                                        index_col=0, sep='\t')
    cell_ids = list(umi_counts_df.columns)
    umi_counts = umi_counts_df.values
    del umi_counts_df
    
    cell_ids_lst.append(cell_ids)
    umi_counts_lst.append(umi_counts)

In [ ]:
pca_projected_lst = []
PCA_COMPS = [2]
if N_PCS_UMAP > 0:
    PCA_COMPS.append(N_PCS_UMAP)
if N_PCS_PHATE > 0:
    PCA_COMPS.append(N_PCS_PHATE)
if N_PCS_DTNE > 0:
    PCA_COMPS.append(N_PCS_DTNE)

PCA_COMPS = np.unique(PCA_COMPS)

logp1_lst = []

for ind_dataset, input_data_folder in enumerate(input_data_folders):
    print(ind_dataset)
    umi_counts = umi_counts_lst[ind_dataset]
    # Perform logp1
    logp1 = do_logp1(umi_counts)

    # Perform PCA to 2 components for visualization and to 10 components for subsequent UMAP
    pca_projected = do_pca(logp1, n_comps_list=PCA_COMPS)
    # del logp1
    logp1_lst.append(logp1)
    pca_projected_lst.append(pca_projected)

In [ ]:
if 'umap' in methods:
    umap_projected_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):   
        print(ind_dataset)
        if N_PCS_UMAP < 0:
            preprocessed = logp1_lst[ind_dataset]
        else:
            preprocessed = pca_projected_lst[ind_dataset][N_PCS_UMAP]
        # Perform UMAP on the larger-number-of-components PCA
        umap_projected = {}
        umap_projected[N_PCS_UMAP] = fit_umap(preprocessed, random_state=None, n_neighbors=15, min_dist=0.1,
                                           n_components=2,
                                           metric='euclidean',
                                           make_plot=False, title='')
        umap_projected_lst.append(umap_projected)

In [ ]:
if 'phate' in methods:
    phate_projected_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):   
        print(ind_dataset)
        # Perform PHATE on the logp1-transformed data
        if N_PCS_PHATE < 0:
            preprocessed = logp1_lst[ind_dataset]
        else:
            preprocessed = pca_projected_lst[ind_dataset][N_PCS_PHATE]
        phate_projected = {}
        phate_projected[N_PCS_PHATE] = fit_phate(preprocessed)
        phate_projected_lst.append(phate_projected)

In [ ]:
if 'DTNE' in methods:
    dtne_projected_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):   
        print(ind_dataset)
        # Perform DTNE on the logp1-transformed data
        if N_PCS_DTNE < 0:
            preprocessed = logp1_lst[ind_dataset]
        else:
            preprocessed = pca_projected_lst[ind_dataset][N_PCS_DTNE]
        dtne_projected = {}
        dtne_projected[N_PCS_DTNE] = fit_DTNE(preprocessed)
        dtne_projected_lst.append(dtne_projected)

## Create pairwise distance plots

In [ ]:
# Read in ground truth squared pairwise distances (divided by the number of dimensions)
true_dists_lst = []
# selected_gene_inds_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders):
    print(ind_dataset)
    orig_counts = umi_counts_lst[ind_dataset]
    num_dims = orig_counts.shape[0]
    true_dists = distance.pdist(orig_counts.T, metric='sqeuclidean')/num_dims
    true_dists_lst.append(true_dists)

In [ ]:
# INCLUDE_SANITY=True
if 'sanity' in methods:
    sanity_lst = []
    for ind_dataset, input_data_folder in enumerate(input_data_folders):
        print(ind_dataset)
        dataset_id = dataset_ids[ind_dataset]
        sanity_path = os.path.join('/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets', dataset_id, 'Sanity')
        sanity_dists = pd.read_csv(os.path.join(sanity_path, 'cell_cell_distance_with_errorbar_avzscore_gt_1.txt'), header=None,
                                   index_col=None, sep='\t').values.flatten()
#         delta_gc_sanity = pd.read_csv(os.path.join(sanity_path, 'delta.txt'), header=None,
#                                     index_col=None, sep='\t').values
#         num_dims = delta_gc_sanity.shape[0]
#         sanity_dists = distance.pdist(delta_gc_sanity.T, metric='sqeuclidean')/num_dims
        sanity_lst.append(sanity_dists)

In [ ]:
# Calculate distances on tree
if 'bonsai' in methods:
    bonsai_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        cell_ids = cell_ids_lst[ind_dataset]
        tree_results = os.path.join(bonsai_results, find_latest_tree_folder(bonsai_results))
        bonsai_dists = get_pdists_on_tree(os.path.join(tree_results, 'tree.nwk'), cell_ids)
        bonsai_dists_lst.append(bonsai_dists)

In [ ]:
if 'pca' in methods:
    pca_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        pca_projected = pca_projected_lst[ind_dataset]
        for n_comps, pca_proj in pca_projected.items():
            if n_comps != 2:
                continue
            pca_dists = distance.pdist(pca_proj.T, metric='sqeuclidean') / 2
            pca_dists_lst.append(pca_dists)

In [ ]:
if 'umap' in methods:
    umap_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        umap_projected = umap_projected_lst[ind_dataset]
        umap_proj = umap_projected[N_PCS_UMAP]
        umap_dists = distance.pdist(umap_proj.T, metric='sqeuclidean') / 2
        umap_dists_lst.append(umap_dists)

In [ ]:
if 'phate' in methods:
    phate_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        phate_projected = phate_projected_lst[ind_dataset]
        phate_proj = phate_projected[N_PCS_PHATE]
        phate_dists = distance.pdist(phate_proj.T, metric='sqeuclidean') / 2
        phate_dists_lst.append(phate_dists)

In [ ]:
if 'DTNE' in methods:
    dtne_dists_lst = []
    for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
        print(ind_dataset)
        dtne_projected = dtne_projected_lst[ind_dataset]
        dtne_proj = dtne_projected[N_PCS_DTNE]
        dtne_dists = distance.pdist(dtne_proj.T, metric='sqeuclidean') / 2
        dtne_dists_lst.append(dtne_dists)

In [ ]:
# Initialize some list for storing information about the different tools
tool_objcts_lst = []

## Plot everything in big figure without the box-plots

In [ ]:
figs_lst = []
axs_lst = []
# ncols = 5 if INCLUDE_SANITY else 4
ncols = len(methods)
for ind_dataset, dataset in enumerate(dataset_descr_lst):
    fig, axs = plt.subplots(nrows=2, ncols=ncols, figsize=(len(methods)*3, 6))
    # Make the 2nd row share y-axis with the first subplot in the 2nd row
    for i in range(1, ncols):  # columns 1 to 3
        axs[1, i].sharex(axs[1, 0])
    figs_lst.append(fig)
    axs_lst.append(axs)
    for ax in axs.flatten():
        ax.set_box_aspect(1)
    # ax.axis('off')
    plt.tight_layout()
    plt.subplots_adjust(left=0, right=1.0, bottom=0.12, top=0.88)
#     plt.tight_layout()
    fig.suptitle(dataset_descr_lst[ind_dataset], fontsize=20)

In [ ]:
tools_lst = ['truth', 'bonsai', 'pca', 'umap', 'phate', 'DTNE']
tools_lst = [tool for tool in tools_lst if tool in methods]
# fig, axs = plt.subplots(ncols=len(tools_lst))
tools_lst = [tool for tool in tools_lst if tool in methods]
bonsai_results = bonsai_results_folders[0]
ind_dataset = 0
true_dists = true_dists_lst[ind_dataset]
dists_lst = {}
if 'truth' in tools_lst:
    dists_lst['truth'] = true_dists_lst
if 'bonsai' in tools_lst:
    dists_lst['bonsai'] = bonsai_dists_lst
if 'pca' in tools_lst:
    dists_lst['pca'] = pca_dists_lst
if 'umap' in tools_lst:
    dists_lst['umap'] = umap_dists_lst
if 'phate' in tools_lst:
    dists_lst['phate'] = phate_dists_lst
if 'DTNE' in tools_lst:
    dists_lst['DTNE'] = dtne_dists_lst


In [ ]:
# Create histogram of correlations figures
tools_lst = ['truth', 'bonsai', 'pca', 'umap', 'phate', 'DTNE']
tools_lst = [tool for tool in tools_lst if tool in methods]

RECALCULATE = True
RECALCULATE = RECALCULATE or not len(tool_objcts_lst)
if RECALCULATE:
    tool_objcts_lst = []
    
# if INCLUDE_SANITY:
#     tools_lst.append('sanity')
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    print("\n\nTreating dataset {}\n".format(dataset_descr_lst[ind_dataset]))
    true_dists = true_dists_lst[ind_dataset]
    tools_dists_dict = {}
    if 'truth' in tools_lst:
        tools_dists_dict['truth'] = true_dists_lst[ind_dataset]
    if 'bonsai' in tools_lst:
        tools_dists_dict['bonsai'] = bonsai_dists_lst[ind_dataset]
    if 'pca' in tools_lst:
        tools_dists_dict['pca'] = pca_dists_lst[ind_dataset]
    if 'umap' in tools_lst:
        tools_dists_dict['umap'] = umap_dists_lst[ind_dataset]
    if 'phate' in tools_lst:
        tools_dists_dict['phate'] = phate_dists_lst[ind_dataset]
    if 'DTNE' in tools_lst:
        tools_dists_dict['DTNE'] = dtne_dists_lst[ind_dataset]
    if RECALCULATE:
        tool_objcts = {}
        true_objct = Dataset(distances=true_dists_lst[ind_dataset], data_type='delta_true', data_id='delta_true', color_types=tools_lst)
        true_objct.true_dataset_ranks = None
        for ind_tool, tool in enumerate(tools_lst):
            data_id = tool + dataset_descr_lst[ind_dataset]
            tool_objcts[tool] = Dataset(distances=tools_dists_dict[tool], data_type=tool, data_id=data_id, color_types=tools_lst)
    else:
        tool_objcts = tool_objcts_lst[ind_dataset]
    if RECALCULATE:
        tool_objcts_lst.append(tool_objcts)

In [ ]:
import matplotlib.ticker as mticker

tools_lst = ['truth', 'bonsai', 'pca', 'umap', 'phate', 'DTNE']
# Make histogram of normalized distances
# fig_hists_lst = []
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    # fig_hists, axs_hists = plt.subplots(ncols=len(tools_lst), nrows=1, sharex=True, sharey=True, figsize=(14,6))
    # fig_hists_lst.append(fig_hists)
    
    for ind_tool, tool in enumerate(tools_lst):
        ax_hist = axs_lst[ind_dataset][1, ind_tool]
        color = tool_objcts_lst[ind_dataset][tool].data_type_color
        dists = dists_lst[tool][ind_dataset]
        dists /= np.mean(dists)
        ax_hist.hist(np.log10(dists), bins=100, density=True, color=color)
        ax_hist.set_title(tool, fontsize=14)
        ax_hist.set_yscale('log')

        ax_hist.xaxis.set_major_locator(mticker.MultipleLocator(3))  # ticks at integers
        ax_hist.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: r'$10^{%d}$' % x))

In [ ]:
# Create Bonsai visualization
# Visualize the tree in the equal-daylight layout, with the correct celltype-annotation
for ind_dataset, bonvis_fig in enumerate(bonvis_fig_lst):
    dataset_fig = figs_lst[ind_dataset]
    dataset_axs = axs_lst[ind_dataset]
    bonvis_fig.bonvis_settings.transf_info.ax_lims = np.array([-1.01, 1.01, -1.01, 1.01])
    # bonvis_fig.update_figure(geometry='flat')
    bonsai_color = tool_objcts_lst[ind_dataset]['bonsai'].data_type_color
    # color = tool_objcts_lst[ind_dataset][tool].data_type_color
    bonvis_fig.bonvis_settings.node_style['annot_info'].annot_to_color['default'] = bonsai_color
    bonvis_fig.remove_artists()
    bonvis_fig.update_colors()
    # bonvis_fig.update_figure(node_style = 'Default')
    bonvis_fig.create_figure(figsize=(6, 6), fig=dataset_fig, ax=dataset_axs[0, 1])
    dataset_descr = dataset_descr_lst[ind_dataset]
    dataset_axs[0,1].axis('off')

In [ ]:
annotation_folders = [os.path.join('/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets', dataset_id, 'annotation') for dataset_id in dataset_ids]

tools_lst = ['pca', 'umap', 'phate', 'DTNE']
tools_lst = [tool for tool in tools_lst if tool in methods]

# Make figure for 2D-PCA and 2D-UMAP
for ind_dataset, input_data_folder in enumerate(input_data_folders):
    print(ind_dataset)
    bonvis_fig = bonvis_fig_lst[ind_dataset]
    # cats_to_color = bonvis_fig.bonvis_settings.node_style['annot_info'].annot_to_color

    proj_dct = {}
    if 'pca' in methods:
        pca_projected = pca_projected_lst[ind_dataset]
        proj_dct['pca'] = pca_projected[2]
    if 'umap' in methods:
        umap_projected = umap_projected_lst[ind_dataset]
        proj_dct['umap'] = umap_projected[N_PCS_UMAP]
    if 'phate' in methods:
        phate_projected = phate_projected_lst[ind_dataset]
        proj_dct['phate'] = phate_projected[N_PCS_PHATE]
    if 'DTNE' in methods:
        dtne_projected = dtne_projected_lst[ind_dataset]
        proj_dct['DTNE'] = dtne_projected[N_PCS_DTNE]
    data_descr = dataset_descr_lst[ind_dataset]
    cell_ids = cell_ids_lst[ind_dataset]
    
    # Read in annotation to get color information for the UMAP
    scData = SCData(onlyObject=True, dataset=dataset_ids[ind_dataset])
    scData.metadata.nCells = len(cell_ids)
    scData.metadata.cellIds = cell_ids
    fig = figs_lst[ind_dataset]

    for ind, tool in enumerate(tools_lst):
        tool_index = ind + 2
    
        ax = axs_lst[ind_dataset][0, tool_index]
        ax.set_box_aspect(1)
    #     ax.axis('off')
        plt.subplots_adjust(left=0, right=1.0, bottom=0, top=1.0)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)

        proj = proj_dct[tool]
        # for n_comps, pca_proj in pca_projected.items():
        #     if n_comps in [2]:
        ax.scatter(proj[0, :], proj[1, :], s=5, color=tool_objcts_lst[ind_dataset][tool].data_type_color)

In [ ]:
# Change some titles and labels
# Create boxplots in separate figure as well, now grouped by dataset
tools_lst = ['truth', 'bonsai', 'pca', 'umap', 'phate', 'DTNE']
tools_lst = [tool for tool in tools_lst if tool in methods]

for ind_fig, fig_obj in enumerate(figs_lst):
    axs = axs_lst[ind_fig]
#     axs[2,0].axis('off')
    if 'truth' in tools_lst:
        axs[1, tools_lst.index('truth')].set_title('Ground truth', fontsize=16)
    if 'bonsai' in tools_lst:
        axs[0, tools_lst.index('bonsai')].set_title('Bonsai', fontsize=16)
        axs[1, tools_lst.index('bonsai')].set_title('Bonsai', fontsize=16)
    if 'pca' in tools_lst:
        axs[0, tools_lst.index('pca')].set_title('PCA', fontsize=16)
        axs[1, tools_lst.index('pca')].set_title('PCA', fontsize=16)
    if 'umap' in tools_lst:
        axs[0, tools_lst.index('umap')].set_title('UMAP', fontsize=16)
        axs[1, tools_lst.index('umap')].set_title('UMAP', fontsize=16)
    if 'phate' in tools_lst:
        axs[0, tools_lst.index('phate')].set_title('PHATE', fontsize=16)
        axs[1, tools_lst.index('phate')].set_title('PHATE', fontsize=16)
    if 'DTNE' in tools_lst:
        axs[0, tools_lst.index('DTNE')].set_title('DTNE', fontsize=16)
        axs[1, tools_lst.index('DTNE')].set_title('DTNE', fontsize=16)
    for ind_ax, ax in enumerate(axs[1,:]):
        ax.set_xlabel("Pairwise distance\n(normalized by mean)", fontsize=14)
    
    # Turn everything off except for the y-label for the top left figure.
    ax = axs[0,0]
    axs[0,0].axis('on')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    ax.tick_params(bottom=False, labelbottom=False)
    ax.tick_params(top=False, right=False)
    ax.tick_params(left=False, labelleft=False)

    # axs[0,0].set_ylabel('Visualization', fontsize=16)
    axs[1,0].set_ylabel('Number of pairs', fontsize=14)
    fig_obj.subplots_adjust(bottom=0.05, top=0.88, left=0.1, right=.9)
    fig_obj.text(0.03, 0.9, 'A)', fontsize=16, fontweight='bold', va='top', ha='left')
    fig_obj.text(0.03, 0.45, 'B)', fontsize=16, fontweight='bold', va='top', ha='left')


In [ ]:
if SAVE_FIG:
    # Save figures
    figure_folder = os.path.join(bnsi_path, 'paper_figure_scripts_and_notebooks','simulated_treelike','overview_figures')
    Path(figure_folder).mkdir(parents=True, exist_ok=True)
    for ind_fig, fig_obj in enumerate(figs_lst):
        fig_obj.savefig(os.path.join(figure_folder, 'random_numbers_dataset_compiled.png'), dpi=300)
        fig_obj.savefig(os.path.join(figure_folder, 'random_numbers_dataset_compiled.svg'))
        print(os.path.join(figure_folder, 'random_numbers_dataset_compiled.svg'))

### Show figures

In [ ]:
ind=0
figs_lst[ind]